# Веб-скрейпинг сайта «Звёздный турнир героев»

Ноутбук для Google Colab. Сайт: https://psdvoretskaya.github.io/mathetmatics/ — математические игры для 2 класса.

Сайт состоит из четырёх страниц: главной и трёх игр. Они устроены по-разному, поэтому в ноутбуке два способа сбора:

- **главная страница статическая** — её текст и ссылки есть в исходном коде, хватает `requests` и `BeautifulSoup`;
- **страницы игр динамические** — разделы и задания рисует JavaScript, поэтому их открывает браузер под управлением Selenium.

**План**

1. Скачать главную страницу и собрать с неё ссылки на игры.
2. Пройти циклом по ссылкам и проверить, что страницы игр динамические.
3. Открыть сессию браузера в Selenium.
4. Циклом по страницам, разделам и заданиям собрать разделы, теорию и условия задач.
5. Проверить результат и посчитать сводные таблицы.
6. Сохранить в CSV, Excel и JSON.

**Этика:** сайт свой; запросы идут с паузами и с `User-Agent`. Ноутбук только открывает задания и читает их текст. Ответы он не вводит, поэтому в журнал результатов сайта ничего не записывается.

## 1. Главная страница: запрос и ссылки

In [ ]:
BASE_URL = "https://psdvoretskaya.github.io/mathetmatics/"

In [ ]:
import os, re, json, time
from urllib.parse import urljoin
import requests
import pandas as pd
from bs4 import BeautifulSoup
from lxml import html as lxml_html

HEADERS = {"User-Agent": "Mozilla/5.0 (study project; Google Colab)"}

def get_page(url):
    """Скачивает страницу и возвращает её исходный код."""
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()                 # остановиться, если сервер ответил ошибкой
    r.encoding = "utf-8"
    return r.text

index_html = get_page(BASE_URL)
soup = BeautifulSoup(index_html, "html.parser")

print("Заголовок:", soup.select_one("h1").get_text(strip=True))
print("Описание:", soup.select_one("main p").get_text(strip=True))

rows = []
for a in soup.select("main a.card"):                       # ссылки-карточки внутри main
    rows.append({
        "Значок": a.select_one("span.ico").get_text(strip=True),
        "Страница": a.select_one("b").get_text(strip=True),
        "Описание": a.select_one("small").get_text(strip=True),
        "Ссылка": urljoin(BASE_URL, a["href"]),             # атрибут href -> полный адрес
    })
pages = pd.DataFrame(rows)

# То же через XPath
tree = lxml_html.fromstring(index_html)
links_xpath = tree.xpath("//a[contains(@class,'card')]/@href")
print("XPath нашёл те же ссылки:", [urljoin(BASE_URL, h) for h in links_xpath] == list(pages["Ссылка"]))
pages

## 2. Цикл по ссылкам: статические страницы или динамические?

Скачиваем каждую страницу игры обычным запросом и ищем в исходном коде карточки разделов (`button.stage`). Если их нет, раздел рисует JavaScript.

In [ ]:
rows = []
for name, url in zip(pages["Страница"], pages["Ссылка"]):
    page = BeautifulSoup(get_page(url), "html.parser")
    rows.append({
        "Страница": name,
        "Заголовок (title)": page.title.get_text(strip=True),
        "Размер, символов": len(str(page)),
        "Карточек разделов в исходном коде": len(page.select("button.stage")),
        "Текст в блоке div#mapGrid": repr(page.select_one("div#mapGrid").get_text(strip=True)),
    })
    time.sleep(1)                        # пауза между запросами
pd.DataFrame(rows)

Карточек разделов в исходном коде нет, блок `div#mapGrid` пустой. Страницы игр динамические, дальше нужен браузер.

## 3. Selenium: сессия браузера

In [ ]:
# Установка Selenium и браузера Chrome в Colab (1–2 минуты)
!pip -q install selenium
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get -qq install -y ./google-chrome-stable_current_amd64.deb > /dev/null

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

options = webdriver.ChromeOptions()
options.add_argument("--headless=new")            # браузер без окна: в Colab экрана нет
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1200,900")

driver = webdriver.Chrome(options=options)        # создаём сессию
wait = WebDriverWait(driver, 15)
print("Браузер запущен")

## 4. Сбор разделов, теории и заданий

Три вспомогательные функции:

- `open_game` открывает страницу игры и вводит имя игрока, которое спрашивает сайт;
- `read_tasks` нажимает по очереди вкладки заданий и читает условие;
- `read_section` открывает раздел, читает описание и теорию, а в разделе с островами проходит по каждому острову.

In [ ]:
def text_of(node):
    """Тег -> текст без лишних пробелов."""
    return " ".join(node.get_text(" ", strip=True).split()) if node else ""

def js_click(element):
    """Клик через JavaScript: нужен для островов, нарисованных в SVG."""
    driver.execute_script("arguments[0].dispatchEvent(new MouseEvent('click', {bubbles: true}))", element)

def open_game(url):
    driver.get(url)                                                   # переход по ссылке
    name = wait.until(EC.visibility_of_element_located((By.CSS_SELECTOR, "input#hello1")))
    name.send_keys("Скрейпер", Keys.ENTER)                            # ввод текста и Enter
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div#mapGrid button.stage")))

def task_text(box, title):
    """Условие задачи из блока #stageBox: у разных разделов оно лежит в разных тегах."""
    if box.select_one("p.task"):
        expression = text_of(box.select_one("div.expr"))                # пример на порядок действий
        return (text_of(box.select_one("p.task")) + " " + expression).strip()
    if box.select_one("div.eqbig"):                                   # уравнения
        return "Реши уравнение: " + text_of(box.select_one("div.eqbig"))
    for node in box.find_all(["p", "div", "span"]):                   # иначе первый длинный текст без вложенных тегов
        if not node.find(True) and len(text_of(node)) >= 15:
            return text_of(node)
    return title                                                      # условие — само название, например «29 · 3»

def read_tasks():
    """Нажимает все вкладки заданий открытого раздела и возвращает [(название, условие, вид)]."""
    result = []
    count = len(driver.find_elements(By.CSS_SELECTOR, "div#secTabs button.ptab"))
    for i in range(count):
        tab = driver.find_elements(By.CSS_SELECTOR, "div#secTabs button.ptab")[i]   # ищем заново: вкладки перерисовываются
        title = re.sub(r"^\d+\.\s*", "", tab.text.strip())
        tab.click()
        wait.until(lambda d: d.find_elements(By.CSS_SELECTOR, "div#secTabs button.ptab")[i].get_attribute("aria-pressed") == "true")
        box = BeautifulSoup(driver.find_element(By.CSS_SELECTOR, "div#stageBox").get_attribute("innerHTML"), "html.parser")
        kind = "аркада" if box.select_one("canvas") else "ввод ответа" if box.select_one("input") else "выбор и нажатия"
        result.append((title, task_text(box, title), kind))
    return result

def read_section(section_id):
    """Открывает раздел и возвращает (описание, теория, [задания])."""
    driver.find_element(By.CSS_SELECTOR, f"button.stage[data-s='{section_id}']").click()
    wait.until(EC.visibility_of_element_located((By.CSS_SELECTOR, "div#secHead h2")))
    head = BeautifulSoup(driver.page_source, "html.parser")
    about = text_of(head.select_one("div#secHead div.whead p"))

    theory = ""
    if head.select_one("button#howBtn"):                              # кнопка «Как это работает?»
        driver.find_element(By.CSS_SELECTOR, "button#howBtn").click()
        how = BeautifulSoup(driver.page_source, "html.parser").select("div.panel.how p")
        theory = " ".join(text_of(p) for p in how)
        driver.find_element(By.CSS_SELECTOR, "button#howOk").click()

    tasks = []
    groups = [g.get("aria-label") or text_of(g.select_one("b")) for g in head.select("div#secGroups [data-g]")]
    if groups:                                                        # раздел с островами: цикл по островам
        for k, group in enumerate(groups):
            js_click(driver.find_elements(By.CSS_SELECTOR, "div#secGroups [data-g]")[k])
            time.sleep(0.2)
            tasks += [(group, *t) for t in read_tasks()]
    else:
        tasks = [("", *t) for t in read_tasks()]

    driver.find_element(By.CSS_SELECTOR, "button#toMap").click()      # назад на карту
    wait.until(EC.visibility_of_element_located((By.CSS_SELECTOR, "div#mapGrid button.stage")))
    return about, theory, tasks

In [ ]:
sections, tasks = [], []

for page_name, url in zip(pages["Страница"], pages["Ссылка"]):       # цикл по страницам
    open_game(url)
    game_map = BeautifulSoup(driver.page_source, "html.parser")
    print(page_name, "—", text_of(game_map.select_one("p#mapSum")))

    for card in game_map.select("div#mapGrid button.stage"):          # цикл по разделам
        if card["data-s"] == "log":                                   # журнал учителя — не раздел с заданиями
            continue
        section = text_of(card.select_one("b"))
        about, theory, found = read_section(card["data-s"])
        sections.append({
            "Страница": page_name,
            "Этап": int(re.search(r"\d+", text_of(card.select_one("span.no"))).group()),
            "Значок": text_of(card.select_one("span.ico")),
            "Раздел": section,
            "Коротко": text_of(card.select_one("small")),
            "Заданий на карточке": int(re.search(r"из (\d+)", text_of(card.select_one("span.cnt"))).group(1)),
            "Заданий собрано": len(found),
            "Описание": about,
            "Теория": theory,
        })
        for n, (group, title, text, kind) in enumerate(found, start=1):
            tasks.append({"Страница": page_name, "Раздел": section, "Блок": group, "№": n,
                          "Задание": title, "Условие": text, "Вид": kind})
    time.sleep(1)                                                     # пауза между страницами

driver.quit()                                                         # закрываем браузер
sections, tasks = pd.DataFrame(sections), pd.DataFrame(tasks)
print("Разделов:", len(sections), "| заданий:", len(tasks))
sections.drop(columns=["Описание", "Теория"])

In [ ]:
tasks.groupby(["Страница", "Раздел"], sort=False).head(2)

## 5. Проверка и сводные таблицы

In [ ]:
print("Число собранных заданий совпадает с числом на карточках:",
      (sections["Заданий на карточке"] == sections["Заданий собрано"]).all())
print("Заданий с пустым условием:", int((tasks["Условие"] == "").sum()))

# Сколько заданий в каждом разделе на каждой странице
by_page = tasks.pivot_table(index="Раздел", columns="Страница", values="Задание", aggfunc="count", fill_value=0, sort=False)
by_page = by_page[list(pages["Страница"])]
display(by_page)

# Какие бывают виды заданий
display(pd.crosstab(tasks["Раздел"], tasks["Вид"]))

# Задания встречаются на нескольких страницах: считаем уникальные
unique_tasks = tasks.drop_duplicates(subset=["Раздел", "Задание", "Условие"])
print("Заданий всего:", len(tasks), "| уникальных:", len(unique_tasks))

# Блоки олимпиадного раздела
blocks = tasks[tasks["Блок"] != ""].drop_duplicates(subset=["Блок", "Задание"]).groupby("Блок", sort=False).size().to_frame("Заданий")
display(blocks)

## 6. Сохранение: CSV, Excel, JSON

In [ ]:
OUT = "turnir_data"
os.makedirs(OUT, exist_ok=True)

tables = {"Страницы": pages, "Разделы": sections, "Задания": tasks,
          "Уникальные задания": unique_tasks, "Задания по страницам": by_page.reset_index()}

with pd.ExcelWriter(f"{OUT}/turnir.xlsx") as xl:
    for name, df in tables.items():
        df.to_csv(f"{OUT}/{name}.csv", index=False, encoding="utf-8-sig")   # utf-8-sig, чтобы Excel открыл кириллицу
        df.to_excel(xl, sheet_name=name, index=False)

with open(f"{OUT}/turnir.json", "w", encoding="utf-8") as f:
    json.dump({name: json.loads(df.to_json(orient="records", force_ascii=False)) for name, df in tables.items()},
              f, ensure_ascii=False, indent=2)

print(sorted(os.listdir(OUT)))

In [ ]:
# Скачать всё одним архивом (только в Colab)
import shutil
shutil.make_archive(OUT, "zip", OUT)
try:
    from google.colab import files
    files.download(f"{OUT}.zip")
except ImportError:
    print("Архив лежит рядом с ноутбуком:", f"{OUT}.zip")